### Extracellular Vesicle (EV) Probability Analysis of Differentially Abundant Proteins
This notebook analyzes the extracellular vesicle (EV) association probability of differentially expressed proteins (DEPs) across different leukemia subtypes. The analysis includes:

Data Loading & Processing: Import differential expression results and merge with EV probability annotations
EV Probability Analysis: Compare EV probabilities between upregulated and downregulated proteins
Statistical Testing: Perform Mann-Whitney U tests to assess significance differences
Visualization: Generate split violin plots showing EV probability distributions by regulation direction
Results Summary: Provide comprehensive statistical summaries and significance testing

The goal is to determine whether upregulated or downregulated proteins in leukemia have different tendencies to be associated with extracellular vesicles, which could provide insights into disease mechanisms and potential biomarker discovery.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import os
import warnings
from scipy.stats import spearmanr
from scipy.stats import pearsonr
from scipy import stats
%matplotlib inline
warnings.filterwarnings('ignore')

In [ ]:
# Define file paths for data and output figures
data_path = os.path.dirname(os.getcwd()) + '/data'
figure_path = os.path.dirname(os.getcwd()) + '/figures'

In [ ]:
# Load differential expression results and merge with EV probability data
datasets = {
    'bcp_all': 'Table S5 BCP-ALL vs Control',
    'aml': 'Table S4 AML vs Control', 
    't_all': 'Table S6 T-ALL vs Control',
    'leukemia': 'Table S3 Leukemia vs Control'
}

# Load protein identifiers mapping table (UniProt IDs to Assay names)
ids = pd.read_excel(data_path + '/results/results.xlsx', sheet_name='Table S2 - Proteins QC', skiprows=1) # supplementary data file
# Load EV probability features for human proteome
features = pd.read_csv(data_path + '/ev/features_human_proteome_w_ev.csv')
ids = ids[['UniProt','Assay']]

# Process differential expression results for each leukemia subtype
processed_data = {}
for name, sheet_name in datasets.items():
    # Read differential expression results
    df = pd.read_excel(data_path + '/results/results.xlsx', sheet_name=sheet_name, skiprows=1)
    
    # Remove duplicate proteins (keep first occurrence)
    df = df.drop_duplicates(subset=['Protein'], keep='first')
    
    # Filter for statistically significant proteins with meaningful fold changes
    # Criteria: adjusted p-value < 0.05 AND absolute log2 fold change >= 1.5
    df = df[(df['adj.P.Val'] < 0.05) & (abs(df['Log2FC']) >= 1.5)]
    
    # Merge with UniProt identifiers
    df = pd.merge(df, ids, left_on='Protein', right_on='Assay', how='inner')
    # Merge with EV probability annotations
    df = pd.merge(df, features, left_on='UniProt', right_on='id', how='inner')
    
    # Count up/down regulated proteins for summary
    upregulated = (df['Log2FC'] >= 1.5).sum()
    downregulated = (df['Log2FC'] <= -1.5).sum()
    
    print(f"{name}: {upregulated} upregulated, {downregulated} downregulated, {len(df)} total")
    
    # Store processed results
    processed_data[name] = df

# Extract individual dataframes for easy access
bcp_all = processed_data['bcp_all']
aml = processed_data['aml']
t_all = processed_data['t_all']
leukemia = processed_data['leukemia']

In [ ]:
plt.style.use('default')
colors = ["#E6393C", "#1F78B4"]
sns.set_palette(colors)

# --- FIXED ORDER (define once) ---
DESIRED_ORDER = ['Leukemia', 'AML', 'T-ALL', 'BCP-ALL']


def prepare_plotting_data(processed_data, desired_order=DESIRED_ORDER):
    """Combine all datasets into a single DataFrame for plotting"""
    plot_data = []

    for dataset_name, df in processed_data.items():
        temp_df = df[['Probability_EV', 'Log2FC']].copy()

        # Ensure EV probability values are within valid range [0,1]
        temp_df['Probability_EV'] = np.clip(temp_df['Probability_EV'], 0, 1)

        # Format dataset names for display with special handling for leukemia
        formatted = dataset_name.replace('_', '-')
        if 'leukemia' in formatted.lower():
            temp_df['Dataset'] = 'Leukemia'
        else:
            temp_df['Dataset'] = formatted.upper()

        # Classify proteins as Increased or Decreased
        temp_df['Regulation'] = np.where(temp_df['Log2FC'] >= 0, 'Increased', 'Decreased')

        plot_data.append(temp_df)

    plot_df = pd.concat(plot_data, ignore_index=True)

    # Optional but helpful: make Dataset an ordered categorical (not required for rest of code)
    plot_df['Dataset'] = pd.Categorical(plot_df['Dataset'], categories=desired_order, ordered=True)

    # Sort so printing/groupby follow the same order
    plot_df = plot_df.sort_values('Dataset')

    return plot_df


def perform_statistical_tests(plot_df, desired_order=DESIRED_ORDER):
    """Perform Mann-Whitney U tests between Increased and Decreased proteins for each dataset"""
    results = []

    # Use explicit desired order (avoids .cat problems)
    for dataset in desired_order:
        dataset_data = plot_df[plot_df['Dataset'] == dataset]
        if dataset_data.empty:
            continue

        up_data = dataset_data[dataset_data['Regulation'] == 'Increased']['Probability_EV']
        down_data = dataset_data[dataset_data['Regulation'] == 'Decreased']['Probability_EV']

        if len(up_data) > 0 and len(down_data) > 0:
            statistic, p_value = stats.mannwhitneyu(up_data, down_data, alternative='two-sided')

            # Effect size 
            effect_size = (np.mean(up_data) - np.mean(down_data)) / np.sqrt(
                (np.std(up_data)**2 + np.std(down_data)**2) / 2
            )

            results.append({
                'Dataset': dataset,
                'n_up': len(up_data),
                'n_down': len(down_data),
                'median_up': np.median(up_data),
                'median_down': np.median(down_data),
                'p_value': p_value,
                'effect_size': effect_size,
                'significant': p_value < 0.05
            })

    return pd.DataFrame(results)

In [ ]:
def add_significance_bars(ax, test_results, desired_order=DESIRED_ORDER):
    """Add statistical significance bars and p-value annotations to the violin plot"""
    for i, dataset in enumerate(desired_order):
        result = test_results[test_results['Dataset'] == dataset]
        if result.empty:
            continue

        p_val = result['p_value'].iloc[0]

        bar_height = 1.05  # above 1.0

        # Draw horizontal significance bar with vertical connectors
        ax.plot([i - 0.2, i + 0.2], [bar_height, bar_height], 'k-', linewidth=1)
        ax.plot([i - 0.2, i - 0.2], [bar_height - 0.01, bar_height], 'k-', linewidth=1)
        ax.plot([i + 0.2, i + 0.2], [bar_height - 0.01, bar_height], 'k-', linewidth=1)

        # Only p-value text (no stars)
        p_text = f'p = {p_val:.3f}' if p_val >= 0.001 else 'p < 0.001'
        ax.text(i, bar_height + 0.03, p_text,
                ha='center', va='bottom',
                fontsize=9, style='italic')

In [ ]:
def create_split_violin_plot(processed_data, desired_order=DESIRED_ORDER):
    plot_df = prepare_plotting_data(processed_data, desired_order=desired_order)
    test_results = perform_statistical_tests(plot_df, desired_order=desired_order)

    fig, ax = plt.subplots(figsize=(10, 4.5))

    sns.violinplot(
        data=plot_df,
        x='Dataset',
        y='Probability_EV',
        hue='Regulation',
        order=desired_order,          # ensures x order
        split=True,
        inner='quart',
        linewidth=1.5,
        ax=ax,
        cut=0
    )

    ax.set_title('', fontsize=16, fontweight='bold', pad=20)
    ax.set_xlabel('', fontsize=14, fontweight='bold')
    ax.set_ylabel('EV Probability', fontsize=14)

    handles, labels = ax.get_legend_handles_labels()
    ax.legend(handles, labels, title='Abundance', title_fontsize=12,
              fontsize=11, loc='upper left', bbox_to_anchor=(1, 1))

    add_significance_bars(ax, test_results, desired_order=desired_order)

    ax.grid(True, alpha=0.3, axis='y')
    ax.set_ylim(0, 1.13)

    # Clip violin plots to max 1.0
    for collection in ax.collections:
        for path in collection.get_paths():
            vertices = path.vertices
            vertices[:, 1] = np.clip(vertices[:, 1], 0, 1)
            path.vertices = vertices

    plt.setp(ax.get_xticklabels(), rotation=0)
    plt.tight_layout()
    plt.show()

    return fig, test_results, plot_df


# ---- Run ----
fig, test_results, plot_df = create_split_violin_plot(processed_data)

print("="*70)
print("STATISTICAL ANALYSIS: UPREGULATED vs DOWNREGULATED")
print("="*70)
print(f"{'Dataset':<15} {'N_Up':<6} {'N_Down':<8} {'Med_Up':<8} {'Med_Down':<10} {'P-value':<10} {'Effect Size':<12} {'Significant'}")
print("-"*70)

for _, row in test_results.iterrows():
    sig_symbol = "***" if row['p_value'] < 0.001 else "**" if row['p_value'] < 0.01 else "*" if row['p_value'] < 0.05 else "ns"
    print(f"{row['Dataset']:<15} {row['n_up']:<6} {row['n_down']:<8} {row['median_up']:<8.3f} {row['median_down']:<10.3f} {row['p_value']:<10.3f} {row['effect_size']:<12.3f} {sig_symbol}")

print("\n" + "="*50)
print("SUMMARY BY DATASET AND REGULATION")
print("="*50)

summary_stats = plot_df.groupby(['Dataset', 'Regulation'])['Probability_EV'].agg([
    'count', 'mean', 'std', 'median',
    lambda x: x.quantile(0.25), lambda x: x.quantile(0.75)
]).round(3)
summary_stats.columns = ['Count', 'Mean', 'Std', 'Median', 'Q25', 'Q75']
print(summary_stats)

print("\n" + "="*50)
print("SIGNIFICANCE LEGEND")
print("="*50)
print("*** : p < 0.001")
print("**  : p < 0.01")
print("*   : p < 0.05")
print("ns  : not significant (p ≥ 0.05)")

fig.savefig(figure_path + '/split_violin_ev_probability_with_significance.png', dpi=300, bbox_inches='tight')
fig.savefig(figure_path + '/split_violin_ev_probability_with_significance.pdf', dpi=300, bbox_inches='tight')
